# 📊 Phase 1: 数据分析与环境验证

本Notebook用于探索性数据分析（EDA）和各模块的交互式验证。

**内容：**
1. 数据下载与质量检查
2. 技术特征可视化
3. 相关性分析与股票筛选
4. 市场机制分析（VIX体制）
5. 交易环境交互测试
6. 基线策略回测
7. 风控模块验证

In [ ]:
import sys, os
sys.path.insert(0, '..')  # if running from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Project modules
from config.settings import CONFIG, DATA_DIR
from data.market_data import MarketDataLoader
from data.universe_screener import UniverseScreener
from features.feature_engineer import FeaturePipeline, TechnicalFeatures, MacroFeatures
from environment.trading_env import TradingEnv
from environment.backtester import BuyAndHoldBenchmark, compute_metrics, compare_strategies
from utils.helpers import set_seed, detect_regime, build_returns_matrix
from utils.risk_manager import RiskManager

set_seed(42)
print('✓ Modules loaded')

## 1. 数据下载

In [ ]:
SYMBOLS = ['AAPL', 'MSFT', 'GOOGL', 'NVDA', 'META', 'JPM', 'JNJ', 'XOM', 'SPY']
START = '2020-01-01'
END   = '2024-12-31'

loader = MarketDataLoader(
    polygon_key=os.getenv('POLYGON_API_KEY', ''),
    finnhub_key=os.getenv('FINNHUB_KEY', ''),
    cache_dir=DATA_DIR / 'cache',
    use_cache=True,
)

market_data = loader.get_ohlcv_universe(SYMBOLS, START, END)
vix = loader.get_vix(START, END)

desc = loader.describe_universe(SYMBOLS, START, END)
print(desc.to_string())

## 2. 价格走势可视化

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 10), facecolor='#0d1117')
fig.suptitle('个股价格走势 (归一化至100)', color='white', fontsize=14, fontweight='bold', y=1.01)

colors = ['#00d4ff','#00ff88','#a855f7','#ff6b35','#fbbf24','#06b6d4','#84cc16','#f43f5e','#94a3b8']
syms_to_plot = [s for s in SYMBOLS if s != 'SPY']
spy_norm = market_data['SPY']['close'] / market_data['SPY']['close'].iloc[0] * 100 if 'SPY' in market_data else None

for i, (ax, sym) in enumerate(zip(axes.flat, syms_to_plot)):
    ax.set_facecolor('#0d1117')
    if sym in market_data and not market_data[sym].empty:
        close = market_data[sym]['close']
        norm = close / close.iloc[0] * 100
        ax.plot(norm.index, norm.values, color=colors[i], linewidth=1.2)
        if spy_norm is not None:
            spy_aligned = spy_norm.reindex(norm.index).ffill()
            ax.plot(spy_aligned.index, spy_aligned.values, color='#475569', linewidth=0.7, linestyle='--', alpha=0.6)
        total_ret = (close.iloc[-1] / close.iloc[0] - 1)
        ax.set_title(f'{sym}  {total_ret:+.0%}', color='white', fontsize=10, fontweight='bold')
    ax.tick_params(colors='#475569', labelsize=7)
    ax.spines[:].set_color('#1e293b')
    ax.axhline(100, color='#334155', linewidth=0.6, linestyle=':')

plt.tight_layout()
plt.savefig('../logs/price_history.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 3. 股票筛选器

In [ ]:
screener = UniverseScreener(
    min_price=5.0,
    min_avg_dollar_volume=5e6,
    min_history_days=252,
)

valid_symbols, report = screener.screen(market_data)
print(f'\n筛选结果: {len(valid_symbols)}/{len(SYMBOLS)} 通过')
print('\n通过的股票:')
passed_report = report[report['passed']][['symbol','avg_price','avg_dollar_vol','annual_vol','sharpe','max_drawdown']]
print(passed_report.to_string(index=False))

## 4. 特征工程

In [ ]:
pipeline = FeaturePipeline()
feature_store = pipeline.build(market_data=market_data, vix=vix)

# 展示AAPL的特征
if 'AAPL' in feature_store:
    aapl_feat = feature_store['AAPL']
    feature_cols = [c for c in aapl_feat.columns if c not in {'open','high','low','close','volume'}]
    print(f'特征数量: {len(feature_cols)}')
    print(f'\n特征统计:')
    print(aapl_feat[feature_cols[:10]].describe().round(3).to_string())

In [ ]:
# 可视化关键技术指标
if 'AAPL' in feature_store and 'AAPL' in market_data:
    df = feature_store['AAPL'].tail(252)  # last 1 year
    raw = market_data['AAPL'].tail(252)

    fig = plt.figure(figsize=(14, 12), facecolor='#0d1117')
    gs = gridspec.GridSpec(4, 1, height_ratios=[3,1,1,1], hspace=0.05)

    # Price + Bollinger
    ax1 = fig.add_subplot(gs[0])
    ax1.set_facecolor('#0d1117')
    ax1.plot(raw.index, raw['close'], color='#00d4ff', linewidth=1.2, label='Close', zorder=3)
    if 'bb_upper' in df.columns:
        # BB bands are normalized, reconstruct from raw data
        tech = TechnicalFeatures()
        raw_with_bb = tech.compute(market_data['AAPL'].tail(252))
        ax1.plot(raw.index, raw_with_bb['bb_upper'].values, color='#a855f7', linewidth=0.7, linestyle='--', alpha=0.7, label='BB Upper')
        ax1.plot(raw.index, raw_with_bb['bb_lower'].values, color='#a855f7', linewidth=0.7, linestyle='--', alpha=0.7, label='BB Lower')
        ax1.fill_between(raw.index, raw_with_bb['bb_upper'].values, raw_with_bb['bb_lower'].values, alpha=0.05, color='#a855f7')
    ax1.set_title('AAPL 技术指标分析', color='white', fontsize=12, fontweight='bold')
    ax1.legend(facecolor='#1e293b', edgecolor='#334155', labelcolor='white', fontsize=8)
    ax1.tick_params(colors='#475569'); ax1.spines[:].set_color('#1e293b')
    ax1.set_xticklabels([])

    # Volume
    ax2 = fig.add_subplot(gs[1])
    ax2.set_facecolor('#0d1117')
    ax2.bar(raw.index, raw['volume'], color='#334155', width=1)
    ax2.set_ylabel('Volume', color='#94a3b8', fontsize=8)
    ax2.tick_params(colors='#475569'); ax2.spines[:].set_color('#1e293b')
    ax2.set_xticklabels([])

    # RSI
    ax3 = fig.add_subplot(gs[2])
    ax3.set_facecolor('#0d1117')
    raw_full = tech.compute(market_data['AAPL'].tail(252))
    ax3.plot(raw.index, raw_full['rsi'].values, color='#fbbf24', linewidth=1)
    ax3.axhline(70, color='#ef4444', linewidth=0.7, linestyle='--')
    ax3.axhline(30, color='#00ff88', linewidth=0.7, linestyle='--')
    ax3.set_ylabel('RSI', color='#94a3b8', fontsize=8)
    ax3.tick_params(colors='#475569'); ax3.spines[:].set_color('#1e293b')
    ax3.set_ylim(0, 100)
    ax3.set_xticklabels([])

    # MACD
    ax4 = fig.add_subplot(gs[3])
    ax4.set_facecolor('#0d1117')
    ax4.plot(raw.index, raw_full['macd'].values, color='#00d4ff', linewidth=1, label='MACD')
    ax4.plot(raw.index, raw_full['macd_signal'].values, color='#ff6b35', linewidth=1, label='Signal')
    colors_hist = ['#00ff88' if v >= 0 else '#ef4444' for v in raw_full['macd_hist'].values]
    ax4.bar(raw.index, raw_full['macd_hist'].values, color=colors_hist, alpha=0.5, width=1)
    ax4.axhline(0, color='#334155', linewidth=0.6)
    ax4.set_ylabel('MACD', color='#94a3b8', fontsize=8)
    ax4.legend(facecolor='#1e293b', edgecolor='#334155', labelcolor='white', fontsize=8)
    ax4.tick_params(colors='#475569', axis='x', rotation=20); ax4.spines[:].set_color('#1e293b')

    plt.savefig('../logs/aapl_indicators.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

## 5. 相关性分析

In [ ]:
from utils.helpers import plot_correlation_heatmap

returns_matrix = build_returns_matrix({s: market_data[s] for s in valid_symbols if s in market_data})
print(f'收益率矩阵: {returns_matrix.shape}')
print(f'平均两两相关系数: {returns_matrix.corr().values[np.triu_indices_from(returns_matrix.corr().values, k=1)].mean():.3f}')

plot_correlation_heatmap(returns_matrix, title='股票收益率相关性矩阵', save_path='../logs/correlation_heatmap.png')

## 6. 市场机制分析

In [ ]:
if 'SPY' in market_data and not vix.empty:
    spy_ret = market_data['SPY']['close'].pct_change().dropna()
    regimes = detect_regime(vix, spy_ret)

    print('市场机制分布:')
    print(regimes['regime'].value_counts().to_string())

    # Plot VIX with regime coloring
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), facecolor='#0d1117', sharex=True)

    regime_colors = {0: '#00d4ff', 1: '#00ff88', 2: '#fbbf24', 3: '#ef4444'}
    vix_aligned = vix.reindex(regimes.index)

    for regime_id, color in regime_colors.items():
        mask = regimes['vix_regime'] == regime_id
        ax1.fill_between(vix_aligned.index, 0, vix_aligned.where(mask), color=color, alpha=0.5)

    ax1.plot(vix_aligned.index, vix_aligned.values, color='white', linewidth=0.8)
    ax1.axhline(20, color='#fbbf24', linewidth=0.6, linestyle='--', alpha=0.5)
    ax1.axhline(30, color='#ff6b35', linewidth=0.6, linestyle='--', alpha=0.5)
    ax1.axhline(40, color='#ef4444', linewidth=0.6, linestyle='--', alpha=0.5)
    ax1.set_ylabel('VIX', color='#94a3b8')
    ax1.set_title('市场机制识别 (VIX体制)', color='white', fontsize=12, fontweight='bold')
    ax1.set_facecolor('#0d1117'); ax1.tick_params(colors='#475569'); ax1.spines[:].set_color('#1e293b')

    # SPY cumulative return
    ax2.set_facecolor('#0d1117')
    spy_cum = (1 + spy_ret).cumprod() * 100
    ax2.plot(spy_cum.index, spy_cum.values, color='#00d4ff', linewidth=1)
    ax2.fill_between(spy_cum.index, 100, spy_cum.values, where=spy_cum.values >= 100, alpha=0.15, color='#00d4ff')
    ax2.fill_between(spy_cum.index, 100, spy_cum.values, where=spy_cum.values < 100, alpha=0.15, color='#ef4444')
    ax2.axhline(100, color='#334155', linewidth=0.7)
    ax2.set_ylabel('SPY (Base=100)', color='#94a3b8')
    ax2.tick_params(colors='#475569', axis='x', rotation=20); ax2.spines[:].set_color('#1e293b')

    plt.savefig('../logs/market_regimes.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

## 7. 交易环境测试

In [ ]:
env_symbols = [s for s in valid_symbols if s != 'SPY' and s in feature_store][:6]

env = TradingEnv(
    feature_store=feature_store,
    symbols=env_symbols,
    initial_capital=1_000_000,
    lookback_window=30,
    episode_length=252,
    reward_type='sharpe',
    render_mode='human',
)

print(f'环境参数:')
print(f'  股票数量:    {env.n}')
print(f'  观测维度:    {env.observation_space.shape}')
print(f'  动作维度:    {env.action_space.shape}')
print(f'  特征数量:    {env.n_features}')
print(f'  交易日数:    {len(env.dates)}')

# 运行一个完整episode（随机动作）
obs, info = env.reset(seed=42)
print(f'\n观测向量统计: mean={obs.mean():.3f}, std={obs.std():.3f}, nan={np.isnan(obs).sum()}')

episode_values = [env.portfolio.total_value]
done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    episode_values.append(info['total_value'])
    done = terminated or truncated

pv = pd.Series(episode_values)
print(f'\n随机策略结果:')
print(f'  总收益: {(pv.iloc[-1]/pv.iloc[0]-1):+.2%}')
print(f'  最大回撤: {info["max_drawdown"]:.2%}')
print(f'  Sharpe: {info["sharpe_ratio"]:.3f}')

## 8. 基线策略回测

In [ ]:
from environment.backtester import compare_strategies, print_comparison_table

# Build price matrix for backtesting
bt_symbols = [s for s in valid_symbols if s != 'SPY']
prices = pd.DataFrame({s: market_data[s]['close'] for s in bt_symbols if s in market_data}).dropna()

# Strategies
bah = BuyAndHoldBenchmark()
bah_values = bah.run(prices)

spy_values = None
if 'SPY' in market_data:
    spy_raw = market_data['SPY']['close'].reindex(prices.index).ffill()
    spy_values = 1_000_000 * spy_raw / spy_raw.iloc[0]

strategies = {'Equal-Weight B&H': bah_values}
if spy_values is not None:
    strategies['SPY'] = spy_values

comp = compare_strategies(strategies, spy_values)
print_comparison_table(comp)

## 9. 风控模块验证

In [ ]:
rm = RiskManager(
    symbols=env_symbols,
    max_position_pct=0.10,
    max_sector_pct=0.30,
    vix_elevated=30.0,
)
rm.peak_value = 1_000_000

# Test matrix: (scenario, weights, vix)
scenarios = [
    ('正常市场 (VIX=18)',      np.ones(len(env_symbols)) * 0.1,  18.0),
    ('VIX升高 (VIX=32)',      np.ones(len(env_symbols)) * 0.12, 32.0),
    ('VIX极端 (VIX=50)',      np.ones(len(env_symbols)) * 0.15, 50.0),
    ('单股过度集中',           np.array([0.5]+[0.0]*(len(env_symbols)-1)), 18.0),
    ('分散化投组',             np.ones(len(env_symbols)) * 0.08,  20.0),
]

print(f'\n{"场景":<20} {"VIX":>6} {"输入权重":>10} {"输出权重":>10} {"缩放比":>8}  调整原因')
print('-' * 80)
for name, weights, vix in scenarios:
    d = rm.evaluate(
        agent_weights=weights,
        portfolio_value=950_000,
        portfolio_value_history=[1_000_000]*5,
        prices=np.array([150.]*len(env_symbols)),
        adv=np.array([1e7]*len(env_symbols)),
        vix=vix,
    )
    scale = d.approved_weights.sum() / (weights.sum() + 1e-8)
    print(f'{name:<20} {vix:>6.0f} {weights.sum():>10.3f} {d.approved_weights.sum():>10.3f} {scale:>8.1%}  {list(d.adjustments.keys())}')

print('\n✓ 风控模块验证完成')

## ✅ Phase 1 完成

**已验证的组件：**
- ✓ 数据采集层（Yahoo Finance + Polygon fallback）
- ✓ 股票筛选器（流动性、价格、数据质量）
- ✓ 特征工程（50+ 技术指标 + 宏观因子）
- ✓ 数据归一化（无未来泄露的滚动z-score）
- ✓ Gymnasium交易环境（观测/动作/奖励）
- ✓ 回测框架（指标计算 + 策略对比）
- ✓ 多层风控系统（VIX + 仓位 + Kelly）

**下一步 Phase 2：**
```python
# 安装RL框架
pip install stable-baselines3 torch

# 开始训练PPO
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv

vec_env = SubprocVecEnv([make_env(feature_store, symbols, seed=i) for i in range(8)])
model = PPO('MlpPolicy', vec_env, learning_rate=3e-4, n_steps=2048, verbose=1)
model.learn(total_timesteps=2_000_000)
```